In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
# Task 1: Write your code here:
df = df = pd.read_csv(f'{path}/Q1_data.csv')

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Display dataset information using info()
df.info()

In [ ]:
# Task 4: Show statistical description using describe()
df.describe()


In [ ]:
# Task 5: Plot the target distribution (delivery_time)
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Drop the 'Order_ID' column from the data
df = df.drop('Order_ID', axis=1)
df

In [ ]:
# Task 2: Handle missing values appropriately

# check missing values
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Handling missing values
# impute missing values of tye clumns = Weather with unknown
df['Weather'] = df['Weather'].fillna("unkown")
df['Weather']

In [ ]:
# impute Traffic_level column missings with mode[0]

df['Traffic_Level'] = df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0])
df['Traffic_Level']

In [ ]:
# impute Time_of_Day with mode()[0]
df['Time_of_Day'] = df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0])
df['Time_of_Day']

In [ ]:
# impute Courier_Experience_yrs with mean
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())
df['Courier_Experience_yrs']

In [ ]:
# impute Delivery_Time with mean
df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].mean())
df['Delivery_Time']

In [ ]:
# Task 3: Check and remove duplicates if any exist
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Encode categorical variables if needed (Bonus if used One Hot Encoding)
# Check
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

df.head()

In [ ]:
# Task 5: Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler

features = df.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

In [ ]:
# Task 1: Split the dataset into features (X) and target (y)
X = df.drop("Delivery_Time", axis=1)
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
# 2: KFold split
from sklearn.model_selection import KFold
from sklearn.metrics import  mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Model
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),

}
all_results = {}

for name in models:
  all_results[name] = {'mae': []}

#Training
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/5")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)

    # Store results
    all_results[model_name]["mae"].append(mae)


for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  Mean Absolute Error:  {np.mean(all_results[model_name]['mae']):.4f}")


In [ ]:
# Task 1: Plot feature importance from your trained model
importances = {}

importances['Random Forest Regressor'] = models['Random Forest Regressor'].feature_importances_

# Create a 1x3 plot
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  plt.barh(features[sorted_idx], imp[sorted_idx])
  plt.title(f"{model_name} Feature Importance")
  plt.xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: